In [1]:
import pandas as pd
import sys

print("Python:", sys.version)
print("Pandas:", pd.__version__)

Python: 3.13.15 (main, Sep  1 2026, 14:16:48) [MSC v.1944 64 bit (AMD64)]
Pandas: 3.0.5


In [2]:
from pathlib import Path

data_path = Path("../data")

csv_files = list(data_path.glob("*.csv"))

for file in csv_files:
    print(file.name)

In [3]:
from pathlib import Path
print(Path.cwd())

c:\Users\user777\Desktop\Conclusion Intelegence project\SpikupCapstone2026_CYRV\notebooks


In [4]:
data_path = Path("../data")

print("Data folder exists:", data_path.exists())

for item in data_path.iterdir():
    print(item.name)

Data folder exists: True
cyrv
README.md


In [5]:
data_path = Path("../data/cyrv")

csv_files = list(data_path.glob("*.csv"))

for file in csv_files:
    print(file.name)

CYRV_customers_dataset.csv
CYRV_geolocation_dataset.csv
CYRV_orders_dataset.csv
CYRV_order_items_dataset.csv
CYRV_order_payments_dataset.csv
CYRV_order_reviews_dataset.csv
CYRV_products_dataset.csv
CYRV_sellers_dataset.csv
product_category_name_translation.csv


In [6]:
file_info = []

for file in csv_files:
    file_info.append({
        "file": file.name,
        "size_mb": round(file.stat().st_size / (1024 * 1024), 2)
    })

pd.DataFrame(file_info).sort_values("size_mb", ascending=False)

,file,size_mb
1,CYRV_geolocation_dataset.csv,59.39
2,CYRV_orders_dataset.csv,16.93
3,CYRV_order_items_dataset.csv,14.83
5,CYRV_order_reviews_dataset.csv,13.78
0,CYRV_customers_dataset.csv,8.71
4,CYRV_order_payments_dataset.csv,5.61
6,CYRV_products_dataset.csv,2.30
7,CYRV_sellers_dataset.csv,0.17
8,product_category_name_translation.csv,0.00


In [7]:
customers = pd.read_csv(data_path / "CYRV_customers_dataset.csv")
orders = pd.read_csv(data_path / "CYRV_orders_dataset.csv")
order_items = pd.read_csv(data_path / "CYRV_order_items_dataset.csv")
payments = pd.read_csv(data_path / "CYRV_order_payments_dataset.csv")
reviews = pd.read_csv(data_path / "CYRV_order_reviews_dataset.csv")
products = pd.read_csv(data_path / "CYRV_products_dataset.csv")
sellers = pd.read_csv(data_path / "CYRV_sellers_dataset.csv")
category_translation = pd.read_csv(
    data_path / "product_category_name_translation.csv"
)

print("Datasets loaded successfully ✅")

Datasets loaded successfully ✅


In [8]:
datasets = {
    "customers": customers,
    "orders": orders,
    "order_items": order_items,
    "payments": payments,
    "reviews": reviews,
    "products": products,
    "sellers": sellers,
    "category_translation": category_translation
}

summary = []

for name, df in datasets.items():
    summary.append({
        "dataset": name,
        "rows": df.shape[0],
        "columns": df.shape[1],
        "missing_values": df.isna().sum().sum(),
        "duplicate_rows": df.duplicated().sum()
    })

summary_df = pd.DataFrame(summary)
summary_df

,dataset,rows,columns,missing_values,duplicate_rows
0,customers,99441,5,0,0
1,orders,99441,8,4908,0
2,order_items,112650,7,0,0
3,payments,103886,5,0,0
4,reviews,99224,7,145903,0
5,products,32951,9,2448,0
6,sellers,3095,4,0,0
7,category_translation,71,2,0,0


In [9]:
for name in ["orders", "reviews", "products"]:
    df = datasets[name]
    
    print(f"\n{name.upper()}")
    print("-" * 40)
    
    missing = df.isna().sum()
    missing = missing[missing > 0]
    
    print(missing)


ORDERS
----------------------------------------
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
dtype: int64

REVIEWS
----------------------------------------
review_comment_title      87656
review_comment_message    58247
dtype: int64

PRODUCTS
----------------------------------------
product_category_name         610
product_name_lenght           610
product_description_lenght    610
product_photos_qty            610
product_weight_g                2
product_length_cm               2
product_height_cm               2
product_width_cm                2
dtype: int64


In [10]:
orders_missing_by_status = (
    orders.groupby("order_status")
    .agg(
        total_orders=("order_id", "size"),
        missing_approved=("order_approved_at", lambda x: x.isna().sum()),
        missing_carrier_date=("order_delivered_carrier_date", lambda x: x.isna().sum()),
        missing_customer_date=("order_delivered_customer_date", lambda x: x.isna().sum())
    )
    .sort_values("total_orders", ascending=False)
)

orders_missing_by_status

,total_orders,missing_approved,missing_carrier_date,missing_customer_date
order_status,,,,
delivered,96478,14,2,8
shipped,1107,0,0,1107
canceled,625,141,550,619
unavailable,609,0,609,609
invoiced,314,0,314,314
processing,301,0,301,301
created,5,5,5,5
approved,2,0,2,2


In [11]:
delivered_issues = orders[
    (orders["order_status"] == "delivered") &
    (
        orders["order_approved_at"].isna() |
        orders["order_delivered_carrier_date"].isna() |
        orders["order_delivered_customer_date"].isna()
    )
]

delivered_issues

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
3002,2d1e2d5bf4dc7227b3bfebb81328c15f,ec05a6d8558c6455f0cbbd8a420ad34f,delivered,2017-11-28 17:44:07,2017-11-28 17:56:40,2017-11-30 18:12:23,NaN,2017-12-18 00:00:00
5323,e04abd8149ef81b95221e88f6ed9ab6a,2127dc6603ac33544953ef05ec155771,delivered,2017-02-18 14:40:00,NaN,2017-02-23 12:04:47,2017-03-01 13:25:33,2017-03-17 00:00:00
16567,8a9adc69528e1001fc68dd0aaebbb54a,4c1ccc74e00993733742a3c786dc3c1f,delivered,2017-02-18 12:45:31,NaN,2017-02-23 09:01:52,2017-03-02 10:05:06,2017-03-21 00:00:00
19031,7013bcfc1c97fe719a7b5e05e61c12db,2941af76d38100e0f8740a374f1a5dc3,delivered,2017-02-18 13:29:47,NaN,2017-02-22 16:25:25,2017-03-01 08:07:38,2017-03-17 00:00:00
20618,f5dd62b788049ad9fc0526e3ad11a097,5e89028e024b381dc84a13a3570decb4,delivered,2018-06-20 06:58:43,2018-06-20 07:19:05,2018-06-25 08:05:00,NaN,2018-07-16 00:00:00
22663,5cf925b116421afa85ee25e99b4c34fb,29c35fc91fc13fb5073c8f30505d860d,delivered,2017-02-18 16:48:35,NaN,2017-02-22 11:23:10,2017-03-09 07:28:47,2017-03-31 00:00:00
23156,12a95a3c06dbaec84bcfb0e2da5d228a,1e101e0daffaddce8159d25a8e53f2b2,delivered,2017-02-17 13:05:55,NaN,2017-02-22 11:23:11,2017-03-02 11:09:19,2017-03-20 00:00:00
26800,c1d4211b3dae76144deccd6c74144a88,684cb238dc5b5d6366244e0e0776b450,delivered,2017-01-19 12:48:08,NaN,2017-01-25 14:56:50,2017-01-30 18:16:01,2017-03-01 00:00:00
38290,d69e5d356402adc8cf17e08b5033acfb,68d081753ad4fe22fc4d410a9eb1ca01,delivered,2017-02-19 01:28:47,NaN,2017-02-23 03:11:48,2017-03-02 03:41:58,2017-03-27 00:00:00
39334,d77031d6a3c8a52f019764e68f211c69,0bf35cac6cc7327065da879e2d90fae8,delivered,2017-02-18 11:04:19,NaN,2017-02-23 07:23:36,2017-03-02 16:15:23,2017-03-22 00:00:00


In [12]:
print("Number of delivered orders with any missing delivery-related date:")
print(len(delivered_issues))

Number of delivered orders with any missing delivery-related date:
23


In [13]:
delivered_issues[
    [
        "order_id",
        "order_status",
        "order_approved_at",
        "order_delivered_carrier_date",
        "order_delivered_customer_date"
    ]
]

,order_id,order_status,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date
3002,2d1e2d5bf4dc7227b3bfebb81328c15f,delivered,2017-11-28 17:56:40,2017-11-30 18:12:23,NaN
5323,e04abd8149ef81b95221e88f6ed9ab6a,delivered,NaN,2017-02-23 12:04:47,2017-03-01 13:25:33
16567,8a9adc69528e1001fc68dd0aaebbb54a,delivered,NaN,2017-02-23 09:01:52,2017-03-02 10:05:06
19031,7013bcfc1c97fe719a7b5e05e61c12db,delivered,NaN,2017-02-22 16:25:25,2017-03-01 08:07:38
20618,f5dd62b788049ad9fc0526e3ad11a097,delivered,2018-06-20 07:19:05,2018-06-25 08:05:00,NaN
22663,5cf925b116421afa85ee25e99b4c34fb,delivered,NaN,2017-02-22 11:23:10,2017-03-09 07:28:47
23156,12a95a3c06dbaec84bcfb0e2da5d228a,delivered,NaN,2017-02-22 11:23:11,2017-03-02 11:09:19
26800,c1d4211b3dae76144deccd6c74144a88,delivered,NaN,2017-01-25 14:56:50,2017-01-30 18:16:01
38290,d69e5d356402adc8cf17e08b5033acfb,delivered,NaN,2017-02-23 03:11:48,2017-03-02 03:41:58
39334,d77031d6a3c8a52f019764e68f211c69,delivered,NaN,2017-02-23 07:23:36,2017-03-02 16:15:23


In [14]:
len(delivered_issues)

23

In [15]:
delivered_issues_check = delivered_issues[
    [
        "order_id",
        "order_approved_at",
        "order_delivered_carrier_date",
        "order_delivered_customer_date"
    ]
].copy()

delivered_issues_check["missing_approved"] = delivered_issues_check["order_approved_at"].isna()
delivered_issues_check["missing_carrier"] = delivered_issues_check["order_delivered_carrier_date"].isna()
delivered_issues_check["missing_customer"] = delivered_issues_check["order_delivered_customer_date"].isna()

delivered_issues_check[
    ["order_id", "missing_approved", "missing_carrier", "missing_customer"]
]

,order_id,missing_approved,missing_carrier,missing_customer
3002,2d1e2d5bf4dc7227b3bfebb81328c15f,False,False,True
5323,e04abd8149ef81b95221e88f6ed9ab6a,True,False,False
16567,8a9adc69528e1001fc68dd0aaebbb54a,True,False,False
19031,7013bcfc1c97fe719a7b5e05e61c12db,True,False,False
20618,f5dd62b788049ad9fc0526e3ad11a097,False,False,True
22663,5cf925b116421afa85ee25e99b4c34fb,True,False,False
23156,12a95a3c06dbaec84bcfb0e2da5d228a,True,False,False
26800,c1d4211b3dae76144deccd6c74144a88,True,False,False
38290,d69e5d356402adc8cf17e08b5033acfb,True,False,False
39334,d77031d6a3c8a52f019764e68f211c69,True,False,False


In [16]:
pattern_summary = (
    delivered_issues_check
    .groupby(
        ["missing_approved", "missing_carrier", "missing_customer"]
    )
    .size()
    .reset_index(name="number_of_orders")
    .sort_values("number_of_orders", ascending=False)
)

pattern_summary

,missing_approved,missing_carrier,missing_customer,number_of_orders
3,True,False,False,14
0,False,False,True,7
1,False,True,False,1
2,False,True,True,1


### Initial Data Quality Findings

- No fully duplicated rows were found in the main datasets.
- Most missing order dates are consistent with the order status.
- 23 delivered orders contain at least one missing delivery-related timestamp:
  - 14 missing approval date only
  - 7 missing customer delivery date only
  - 1 missing carrier date only
  - 1 missing both carrier and customer delivery dates
- Review title and message fields contain many missing values because written comments are optional.
- Product data contains 610 records with missing category/listing information and 2 records with missing physical dimensions.

In [17]:
orders.dtypes

order_id                         str
customer_id                      str
order_status                     str
order_purchase_timestamp         str
order_approved_at                str
order_delivered_carrier_date     str
order_delivered_customer_date    str
order_estimated_delivery_date    str
dtype: object

In [18]:
date_columns = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]

for col in date_columns:
    orders[col] = pd.to_datetime(orders[col], errors="coerce")

orders[date_columns].dtypes

order_purchase_timestamp         datetime64[us]
order_approved_at                datetime64[us]
order_delivered_carrier_date     datetime64[us]
order_delivered_customer_date    datetime64[us]
order_estimated_delivery_date    datetime64[us]
dtype: object

In [19]:
print("First order:", orders["order_purchase_timestamp"].min())
print("Last order:", orders["order_purchase_timestamp"].max())

First order: 2016-09-04 21:15:19
Last order: 2018-10-17 17:30:18


In [20]:
monthly_orders = (
    orders
    .assign(month=orders["order_purchase_timestamp"].dt.to_period("M"))
    .groupby("month")
    .size()
    .reset_index(name="number_of_orders")
)

monthly_orders

,month,number_of_orders
0,2016-09,4
1,2016-10,324
2,2016-12,1
3,2017-01,800
4,2017-02,1780
5,2017-03,2682
6,2017-04,2404
7,2017-05,3700
8,2017-06,3245
9,2017-07,4026


In [21]:
orders[
    orders["order_purchase_timestamp"] >= "2018-08-01"
]["order_purchase_timestamp"].dt.date.value_counts().sort_index()

order_purchase_timestamp
2018-08-01    311
2018-08-02    302
2018-08-03    314
2018-08-04    245
2018-08-05    276
2018-08-06    372
2018-08-07    370
2018-08-08    316
2018-08-09    289
2018-08-10    256
2018-08-11    188
2018-08-12    197
2018-08-13    292
2018-08-14    316
2018-08-15    288
2018-08-16    320
2018-08-17    257
2018-08-18    198
2018-08-19    204
2018-08-20    256
2018-08-21    243
2018-08-22    187
2018-08-23    144
2018-08-24     99
2018-08-25     69
2018-08-26     73
2018-08-27     67
2018-08-28     44
2018-08-29     14
2018-08-30      4
2018-08-31      1
2018-09-03      4
2018-09-06      3
2018-09-10      1
2018-09-11      1
2018-09-12      1
2018-09-13      1
2018-09-17      1
2018-09-20      1
2018-09-25      1
2018-09-26      1
2018-09-29      1
2018-10-01      1
2018-10-03      1
2018-10-16      1
2018-10-17      1
Name: count, dtype: int64

In [22]:
orders_core = orders[
    (orders["order_purchase_timestamp"] >= "2017-01-01") &
    (orders["order_purchase_timestamp"] < "2018-09-01")
].copy()

print("Core analysis period:")
print(orders_core["order_purchase_timestamp"].min())
print("to")
print(orders_core["order_purchase_timestamp"].max())

print("\nOrders included:", len(orders_core))

Core analysis period:
2017-01-05 11:56:06
to
2018-08-31 16:13:44

Orders included: 99092


In [23]:
payment_per_order = (
    payments
    .groupby("order_id", as_index=False)["payment_value"]
    .sum()
)

orders_revenue = orders_core.merge(
    payment_per_order,
    on="order_id",
    how="left"
)

print("Orders:", len(orders_revenue))
print("Orders without payment:", orders_revenue["payment_value"].isna().sum())

Orders: 99092
Orders without payment: 0


In [24]:
orders_revenue["month"] = (
    orders_revenue["order_purchase_timestamp"]
    .dt.to_period("M")
)

monthly_sales = (
    orders_revenue
    .groupby("month")
    .agg(
        number_of_orders=("order_id", "nunique"),
        gross_payment_value=("payment_value", "sum"),
        average_order_value=("payment_value", "mean")
    )
    .reset_index()
)

monthly_sales["gross_payment_value"] = (
    monthly_sales["gross_payment_value"].round(2)
)

monthly_sales["average_order_value"] = (
    monthly_sales["average_order_value"].round(2)
)

monthly_sales

,month,number_of_orders,gross_payment_value,average_order_value
0,2017-01,800,138488.04,173.11
1,2017-02,1780,291908.01,163.99
2,2017-03,2682,449863.60,167.73
3,2017-04,2404,417788.03,173.79
4,2017-05,3700,592918.82,160.25
5,2017-06,3245,511276.38,157.56
6,2017-07,4026,592382.92,147.14
7,2017-08,4331,674396.32,155.71
8,2017-09,4285,727762.45,169.84
9,2017-10,4631,779677.88,168.36


In [25]:
monthly_sales["order_growth_pct"] = (
    monthly_sales["number_of_orders"]
    .pct_change() * 100
).round(2)

monthly_sales["payment_growth_pct"] = (
    monthly_sales["gross_payment_value"]
    .pct_change() * 100
).round(2)

monthly_sales

,month,number_of_orders,gross_payment_value,average_order_value,order_growth_pct,payment_growth_pct
0,2017-01,800,138488.04,173.11,NaN,NaN
1,2017-02,1780,291908.01,163.99,122.50,110.78
2,2017-03,2682,449863.60,167.73,50.67,54.11
3,2017-04,2404,417788.03,173.79,-10.37,-7.13
4,2017-05,3700,592918.82,160.25,53.91,41.92
5,2017-06,3245,511276.38,157.56,-12.30,-13.77
6,2017-07,4026,592382.92,147.14,24.07,15.86
7,2017-08,4331,674396.32,155.71,7.58,13.84
8,2017-09,4285,727762.45,169.84,-1.06,7.91
9,2017-10,4631,779677.88,168.36,8.07,7.13


In [26]:
print("Top 5 months by number of orders:")
display(
    monthly_sales.nlargest(5, "number_of_orders")[
        ["month", "number_of_orders", "gross_payment_value", "average_order_value"]
    ]
)

print("\nTop 5 months by gross payment value:")
display(
    monthly_sales.nlargest(5, "gross_payment_value")[
        ["month", "number_of_orders", "gross_payment_value", "average_order_value"]
    ]
)

Top 5 months by number of orders:


,month,number_of_orders,gross_payment_value,average_order_value
10,2017-11,7544,1194882.80,158.39
12,2018-01,7269,1115004.18,153.39
14,2018-03,7211,1159652.12,160.82
15,2018-04,6939,1160785.48,167.28
16,2018-05,6873,1153982.15,167.90



Top 5 months by gross payment value:


,month,number_of_orders,gross_payment_value,average_order_value
10,2017-11,7544,1194882.80,158.39
15,2018-04,6939,1160785.48,167.28
14,2018-03,7211,1159652.12,160.82
16,2018-05,6873,1153982.15,167.90
12,2018-01,7269,1115004.18,153.39


## Initial Sales Trends Findings

- November 2017 recorded the highest order volume (7,544 orders) and the highest gross payment value (~1.19M).
- High sales months appear to be driven mainly by order volume rather than major changes in average order value.
- April 2018 generated slightly higher gross payment value than March 2018 despite having fewer orders, supported by a higher average order value.

In [27]:
delivered_orders = orders[
    orders["order_status"] == "delivered"
].copy()

delivered_orders["actual_delivery_days"] = (
    delivered_orders["order_delivered_customer_date"]
    - delivered_orders["order_purchase_timestamp"]
).dt.total_seconds() / 86400

delivered_orders["delay_days"] = (
    delivered_orders["order_delivered_customer_date"]
    - delivered_orders["order_estimated_delivery_date"]
).dt.total_seconds() / 86400

delivered_orders["on_time"] = (
    delivered_orders["order_delivered_customer_date"]
    <= delivered_orders["order_estimated_delivery_date"]
)

print("Delivered orders:", len(delivered_orders))
print(
    "Average actual delivery days:",
    round(delivered_orders["actual_delivery_days"].mean(), 2)
)
print(
    "On-time delivery rate:",
    round(delivered_orders["on_time"].mean() * 100, 2),
    "%"
)

Delivered orders: 96478
Average actual delivery days: 12.56
On-time delivery rate: 91.88 %


In [28]:
late_orders = delivered_orders[
    delivered_orders["on_time"] == False
].copy()

print("Late orders:", len(late_orders))

print(
    "Late delivery rate:",
    round(len(late_orders) / len(delivered_orders) * 100, 2),
    "%"
)

print(
    "Average delay days for late orders:",
    round(late_orders["delay_days"].mean(), 2)
)

print(
    "Maximum delay days:",
    round(late_orders["delay_days"].max(), 2)
)

Late orders: 7834
Late delivery rate: 8.12 %
Average delay days for late orders: 9.55
Maximum delay days: 188.98


In [29]:
late_orders["delay_days"].describe(
    percentiles=[0.5, 0.75, 0.9, 0.95, 0.99]
)

count    7826.000000
mean        9.551776
std        13.952540
min         0.002500
50%         5.806481
75%        11.820978
90%        21.536088
95%        29.639578
99%        62.786780
max       188.975081
Name: delay_days, dtype: float64

In [30]:
print("Late orders with missing delay_days:")
display(
    late_orders[
        late_orders["delay_days"].isna()
    ][[
        "order_id",
        "order_purchase_timestamp",
        "order_delivered_customer_date",
        "order_estimated_delivery_date"
    ]]
)

print("\nTop 10 longest delays:")
display(
    late_orders[
        ["order_id", "delay_days"]
    ]
    .sort_values("delay_days", ascending=False)
    .head(10)
)

Late orders with missing delay_days:


,order_id,order_purchase_timestamp,order_delivered_customer_date,order_estimated_delivery_date
3002,2d1e2d5bf4dc7227b3bfebb81328c15f,2017-11-28 17:44:07,NaT,2017-12-18
20618,f5dd62b788049ad9fc0526e3ad11a097,2018-06-20 06:58:43,NaT,2018-07-16
43834,2ebdfc4f15f23b91474edf87475f108e,2018-07-01 17:05:11,NaT,2018-07-30
79263,e69f75a717d64fc5ecdfae42b2e8e086,2018-07-01 22:05:55,NaT,2018-07-30
82868,0d3268bad9b086af767785e3f0fc0133,2018-07-01 21:14:02,NaT,2018-07-24
92643,2d858f451373b04fb5c984a1cc2defaf,2017-05-25 23:22:43,NaT,2017-06-23
97647,ab7c89dc1bf4a1ead9d6ec1ec8968a84,2018-06-08 12:09:39,NaT,2018-06-26
98038,20edc82cf5400ce95e1afacc25798b31,2018-06-27 16:09:12,NaT,2018-07-19



Top 10 longest delays:


,order_id,delay_days
55619,1b3190b2dfa9d789e1f14c05b647a14a,188.975081
19590,ca07593549f1816d26a572e06dc1eab6,181.608785
11399,47b40429ed8cce3aee9199792275433f,175.869109
81401,2fe324febf907e3ea3f2aa9650869fa5,167.708414
89130,285ab9426d6982034523a855f55a885e,166.583380
61610,440d0d17af552815d15a9e41abe49359,165.633912
68769,c27815f7e3dd0b926b58552628481575,162.718345
40847,d24e8541128cea179a11a65176e0a96f,161.775336
38509,0f4519c5f1c541ddec9f21b3bddd533a,161.609965
54480,2d7561026d542c8dbd8f0daeadf67a43,159.609931


In [31]:
delivery_reviews = delivered_orders.merge(
    reviews[["order_id", "review_score"]],
    on="order_id",
    how="inner"
)

review_comparison = (
    delivery_reviews
    .groupby("on_time")
    .agg(
        number_of_orders=("order_id", "count"),
        average_review_score=("review_score", "mean")
    )
    .reset_index()
)

review_comparison["delivery_status"] = review_comparison["on_time"].map({
    True: "On time",
    False: "Late"
})

review_comparison[
    ["delivery_status", "number_of_orders", "average_review_score"]
]

,delivery_status,number_of_orders,average_review_score
0,Late,7708,2.568500
1,On time,88653,4.293718


In [32]:
review_distribution = pd.crosstab(
    delivery_reviews["review_score"],
    delivery_reviews["delivery_status"],
    normalize="columns"
) * 100

review_distribution.round(2)

KeyError: 'delivery_status'

In [36]:
delivery_reviews["delivery_status"] = delivery_reviews["on_time"].replace({
    True: "On time",
    False: "Late"
})

review_distribution = pd.crosstab(
    delivery_reviews["review_score"],
    delivery_reviews["delivery_status"],
    normalize="columns"
) * 100

review_distribution.round(2)

delivery_status,Late,On time
review_score,,
1,46.12,6.60
2,7.86,2.63
3,11.35,7.99
4,12.38,20.34
5,22.29,62.43


In [37]:
late_with_reviews = delivery_reviews[
    (delivery_reviews["on_time"] == False) &
    (delivery_reviews["delay_days"].notna())
].copy()

late_with_reviews["delay_group"] = pd.cut(
    late_with_reviews["delay_days"],
    bins=[0, 3, 7, 14, 30, float("inf")],
    labels=["0-3 days", "4-7 days", "8-14 days", "15-30 days", "30+ days"],
    include_lowest=True
)

delay_review_summary = (
    late_with_reviews
    .groupby("delay_group", observed=False)
    .agg(
        number_of_orders=("order_id", "count"),
        average_review_score=("review_score", "mean")
    )
    .reset_index()
)

delay_review_summary

,delay_group,number_of_orders,average_review_score
0,0-3 days,2651,3.765372
1,4-7 days,1777,2.316263
2,8-14 days,1758,1.749147
3,15-30 days,1169,1.617622
4,30+ days,345,2.023188


In [38]:
delay_star_distribution = pd.crosstab(
    late_with_reviews["delay_group"],
    late_with_reviews["review_score"],
    normalize="index"
) * 100

delay_star_distribution.round(2)

review_score,1,2,3,4,5
delay_group,,,,,
0-3 days,13.81,5.32,14.52,23.24,43.12
4-7 days,53.01,8.33,10.80,9.74,18.12
8-14 days,67.92,10.07,9.22,4.78,8.02
15-30 days,71.09,10.61,8.90,4.28,5.13
30+ days,64.06,4.64,9.28,8.99,13.04


### Delivery Delay & Customer Satisfaction
Customer satisfaction deteriorates sharply once delivery delays exceed three days. Only 13.8% of orders delayed by 0–3 days received a 1-star review, compared with 53.0% for delays of 4–7 days and 71.1% for delays of 15–30 days. This suggests that minimizing delays beyond three days could be particularly important for customer satisfaction.